In [31]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "data"
INPUT_JSONL = DATA_DIR / "extracted" / "extracted_desc_20260407_023403.jsonl"
df = pd.read_json(INPUT_JSONL, orient="records", lines=True)

In [32]:
# 将df的extracted列展开为多列，并将这些列添加到df中，最后删除extracted列
extracted_df = df["extracted"].apply(pd.Series)
processed_df = pd.concat([df.drop(columns=["extracted"]), extracted_df], axis=1)
# 把status列取值为skipped_empty_desc的行去掉，便于观察
processed_df = processed_df[processed_df["status"] != "skipped_empty_desc"]
processed_df.head(5)

,id,name,fullDesc,status,material,pattern,shape,state,color,craft,raw_excavation_info,raw_size_in_desc,other_info
0,05EB474CCFC646D6A4B7184EB047DA09,新石器时代大汶口文化灰陶环,暂无描述,success,灰陶,None,None,None,None,None,None,None,None
1,424ef49f0ed143fe8834415acda7f1e7,新石器时代大汶口文化黑陶高柄杯,暂无描述,success,None,None,高柄,None,黑陶,None,None,None,None
2,ABE33DEB2BF94D3B95A313E93F60DB71,新石器时代大汶口文化灰陶罐,暂无描述,success,灰陶,None,罐,None,None,None,None,None,None
3,88EBDFF0B8B7497995CB37BA5417A033,新石器时代大汶口文化红褐陶器盖,暂无描述,success,红褐陶,None,None,None,None,None,None,None,None
4,060D1DCF727446EABCC62996C9357A29,新石器时代大汶口文化白陶器,暂无描述,success,白陶,None,器,None,None,None,None,None,None


In [33]:
processed_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4119 entries, 0 to 4118
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   id                   4119 non-null   object
 1   name                 4119 non-null   object
 2   fullDesc             4119 non-null   object
 3   status               4119 non-null   object
 4   material             2842 non-null   object
 5   pattern              299 non-null    object
 6   shape                2635 non-null   object
 7   state                96 non-null     object
 8   color                1208 non-null   object
 9   craft                297 non-null    object
 10  raw_excavation_info  103 non-null    object
 11  raw_size_in_desc     82 non-null     object
 12  other_info           266 non-null    object
dtypes: object(13)
memory usage: 418.5+ KB


In [46]:
# 将提取数据与原始数据按id列合并
ORIGIN_DATA = DATA_DIR / "processed_df.jsonl"
origin_df = pd.read_json(ORIGIN_DATA, orient="records", lines=True)
processed_df.drop(columns=["name","fullDesc","status"], inplace=True)
df_merged = pd.merge(origin_df, processed_df, on="id", how="inner")
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4119 entries, 0 to 4118
Data columns (total 42 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    4119 non-null   object 
 1   name                  4119 non-null   object 
 2   era                   3915 non-null   object 
 3   culture               3594 non-null   object 
 4   time                  4119 non-null   object 
 5   dimensionsDesc        3744 non-null   object 
 6   structuredDimensions  0 non-null      float64
 7   fullDesc              518 non-null    object 
 8   features              0 non-null      float64
 9   excavationLocation    0 non-null      float64
 10  currentLocation       4119 non-null   object 
 11  excavationDate        0 non-null      float64
 12  sourceCitation        4119 non-null   object 
 13  collectionInfo        4119 non-null   object 
 14  images                4119 non-null   object 
 15  shape_type           

In [ ]:
from pprint import pprint
features = ['material', 'pattern', 'shape', 'state', 'color', 'craft',
       'raw_excavation_info', 'raw_size_in_desc', 'other_info']
for feature in features:
    print(f"{feature}的取值分布：")
    pprint(df_merged[feature].value_counts())
    print("\n")

In [ ]:
# 看一下raw_size_in_desc非空的行，对比这些行的raw_size_in_desc和dimensionsDesc的取值，观察相关性
size_vs_desc = df_merged[df_merged["raw_size_in_desc"].notna()][["id","name","fullDesc","raw_size_in_desc", "dimensionsDesc"]]
# 直接导出csv，对两边进行人工校准，再重新导入并覆盖

# 太麻烦了，开摆，不要弄这么复杂了，直接用dimensionsDesc提结构化尺寸信息

In [54]:
df_merged.head(5)

,id,name,era,culture,time,dimensionsDesc,structuredDimensions,fullDesc,features,excavationLocation,...,yearName,material,pattern,shape,state,color,craft,raw_excavation_info,raw_size_in_desc,other_info
0,003895c08aa84c2a98d3d1696db7ffcf,新石器时代大汶口文化红陶鋬盆,新石器时代,大汶口文化,"{'year': None, 'month': None, 'day': None}",口径19.2,NaN,夹砂红陶，侈口，上腹斜置，折腹后收为小平底，折腹处印压出一周斜点纹，鋬手由中腹弯出，端部窄薄上翘。,NaN,NaN,...,新石器时代,夹砂红陶,折腹处印压出一周斜点纹,侈口，上腹斜置，折腹后收为小平底，鋬手由中腹弯出，端部窄薄上翘,None,None,None,None,None,None
1,0043E78E1BCD4F11902DCA3C76607652,新石器时代大汶口文化褐陶鬶,新石器时代,大汶口文化,"{'year': None, 'month': None, 'day': None}",口径10.5厘米,NaN,None,NaN,NaN,...,新石器时代,褐陶,None,鬶,None,None,None,None,None,None
2,004658C82423474EB46AA00F6BEEDAFC,新石器时代大汶口文化红陶壶,新石器时代,大汶口文化,"{'year': None, 'month': None, 'day': None}",高*口径*底径：15.3*7.5*6.2,NaN,None,NaN,NaN,...,新石器时代,红陶,None,None,None,None,None,None,None,None
3,0053DF2D67D346059E75B0729A69FE5B,新石器时代夹砂黑陶碗,新石器时代,None,"{'year': None, 'month': None, 'day': None}",口径13，底径7，高5,NaN,None,NaN,NaN,...,新石器时代,夹砂黑陶,None,碗,None,黑陶,None,None,None,None
4,005C79C1C40F4AB9AA9183E5B383E158,新石器时代龙山文化陶褐鬹,新石器时代,龙山文化,"{'year': None, 'month': None, 'day': None}",通高*腹围*口径*流长：39.5*37*9.1*9.9,NaN,None,NaN,NaN,...,新石器时代,None,None,None,None,None,None,None,None,None


In [55]:
import pandas as pd

# 定义需要提取的属性列
feature_cols = ['material', 'pattern', 'shape', 'state', 'color', 'craft']

def build_features(row):
    features = []
    for col in feature_cols:
        val = row.get(col)
        # 过滤掉 NaN、None 以及空字符串
        if pd.notna(val) and str(val).strip() != '':
            features.append({
                "label": col,
                "value": str(val).strip()
            })
    # 如果全为空，可以选择返回空列表 [] 或 None，这里返回 [] 保持结构统一
    return features if features else []

# 批量应用到 features 列
df_merged['features'] = df_merged.apply(build_features, axis=1)

print("✅ features 列组装完毕！前 3 行预览：")
print(df_merged['features'].head(3).tolist())

✅ features 列组装完毕！前 3 行预览：
[[{'label': 'material', 'value': '夹砂红陶'}, {'label': 'pattern', 'value': '折腹处印压出一周斜点纹'}, {'label': 'shape', 'value': '侈口，上腹斜置，折腹后收为小平底，鋬手由中腹弯出，端部窄薄上翘'}], [{'label': 'material', 'value': '褐陶'}, {'label': 'shape', 'value': '鬶'}], [{'label': 'material', 'value': '红陶'}]]


In [ ]:
# 太累了，下面直接贴ai代码

In [57]:
import os
import json
import asyncio
import logging
from datetime import datetime
from pathlib import Path
import pandas as pd

# 必须执行此操作，否则在 Jupyter 中运行高并发 async 代码会引发 RuntimeError: This event loop is already running
import nest_asyncio
nest_asyncio.apply()

from dotenv import load_dotenv
from openai import AsyncOpenAI
from tqdm.asyncio import tqdm

# 1. 路径定义 (当前 Notebook 在 wwsdw.net/src 下)
BASE_DIR = Path.cwd().parent 
ENV_FILE = BASE_DIR / ".env"
OUTPUT_DIR = BASE_DIR / "data" / "extracted"
LOG_DIR = BASE_DIR / "log"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# 2. 生成带时间戳的文件名
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_FILE = OUTPUT_DIR / f"dimensions_{timestamp}.jsonl"
LOG_FILE = LOG_DIR / f"api_dimensions_{timestamp}.log"

# 3. 日志配置
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE, encoding="utf-8"),
        logging.StreamHandler()
    ]
)
logging.getLogger("httpx").setLevel(logging.WARNING)

# 4. 初始化大模型客户端
load_dotenv(ENV_FILE)
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com/v1")

if not DEEPSEEK_API_KEY:
    raise ValueError(f"未找到 API KEY，请检查 {ENV_FILE}")

client = AsyncOpenAI(api_key=DEEPSEEK_API_KEY, base_url=DEEPSEEK_BASE_URL)

print("✅ 单元格 1 运行成功：环境、路径与日志配置完毕。")

✅ 单元格 1 运行成功：环境、路径与日志配置完毕。


In [58]:
# ==========================================
# 核心 Prompt 设计
# ==========================================
SYSTEM_PROMPT = """你是一个专业的考古数据结构化专家。
你的任务是将人类书写的、非结构化的文物尺寸描述，提取为结构化的 JSON 数组。

提取规则：
1. 你的返回必须是一个包含 "dimensions" 键的 JSON 对象，该键对应一个列表。
2. 列表中的每个对象必须严格包含以下 5 个字段：unit, value, deviation, range, label。
3. 如果未明确标明单位，默认 unit 为 "cm"。
4. 遇到类似 "直径*高=10*20" 的复合描述，请利用逻辑推理按顺序对应拆解：一个对象 label="直径", value=10.0；另一个对象 label="高度", value=20.0。
5. 对于区间值（如 "高10-15"），设置 value=null，range=[10.0, 15.0]。
6. 对于固定值加偏差（如 "口径10±0.1"），设置 value=10.0, deviation=0.1。
7. 空缺的数值字段请填 null。

输出格式示例：
{
  "dimensions": [
    {"unit": "cm", "value": 54.0, "deviation": null, "range": null, "label": "通高"},
    {"unit": "cm", "value": 37.0, "deviation": null, "range": null, "label": "口径"}
  ]
}
请严格输出 JSON，不要附加任何解释或 markdown 标记。"""

async def extract_dimension(item_id: str, desc: str, sem: asyncio.Semaphore, file_lock: asyncio.Lock):
    async with sem:
        try:
            response = await client.chat.completions.create(
                model="deepseek-chat",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": f"提取以下尺寸描述:\n{desc}"}
                ],
                response_format={"type": "json_object"},
                temperature=0.1,
                timeout=45
            )
            
            extracted_json = json.loads(response.choices[0].message.content)
            result_data = {"id": item_id, "structuredDimensions": extracted_json.get("dimensions", [])}
            
            async with file_lock:
                with open(OUTPUT_FILE, "a", encoding="utf-8") as f:
                    f.write(json.dumps(result_data, ensure_ascii=False) + "\n")
            
            logging.info(f"Success {item_id}")
            
        except Exception as e:
            logging.error(f"Failed {item_id}: {str(e)}")
            # 失败兜底，确保后续合并不会错位
            async with file_lock:
                with open(OUTPUT_FILE, "a", encoding="utf-8") as f:
                    f.write(json.dumps({"id": item_id, "structuredDimensions": []}, ensure_ascii=False) + "\n")

async def run_extraction():
    # 筛选出有尺寸描述的行
    valid_mask = df_merged["dimensionsDesc"].notna() & (df_merged["dimensionsDesc"].str.strip() != "")
    tasks_data = df_merged[valid_mask][["id", "dimensionsDesc"]].to_dict("records")
    
    if not tasks_data:
        print("没有需要提取尺寸的数据！")
        return
        
    print(f"准备处理 {len(tasks_data)} 条尺寸数据...")
    sem = asyncio.Semaphore(20) # 控制并发为20
    file_lock = asyncio.Lock()
    
    tasks = [extract_dimension(row["id"], row["dimensionsDesc"], sem, file_lock) for row in tasks_data]
    await tqdm.gather(*tasks, desc="提取尺寸进度")
    print(f"\n✅ 所有请求完成！原始结果已存入: {OUTPUT_FILE}")

# 在 Jupyter 中直接 await 运行异步主函数
await run_extraction()

准备处理 3744 条尺寸数据...


提取尺寸进度:   0%|          | 1/3744 [00:02<2:17:44,  2.21s/it]2026-04-07 05:44:29,217 - INFO - Success 080647CB24DA4C8783EC62742E2B60FE
2026-04-07 05:44:29,221 - INFO - Success 525A71D96DB146C99D1D53672F7BE4C5
2026-04-07 05:44:29,232 - INFO - Success 334AD65BEC874925A2A06C604758B4BA
2026-04-07 05:44:29,291 - INFO - Success 9b10d234ef5b458388852a47d11cd8ee
提取尺寸进度:   0%|          | 5/3744 [00:02<22:26,  2.78it/s]  2026-04-07 05:44:29,422 - INFO - Success FC8806F3E8C04AD1B1132C6F38A224C8
2026-04-07 05:44:29,443 - INFO - Success 522A7D1C067C4F73BD94D0A3EABB294E
提取尺寸进度:   0%|          | 7/3744 [00:02<15:43,  3.96it/s]2026-04-07 05:44:29,722 - INFO - Success 333D1809472D4122AF4BAEFA39F58674
2026-04-07 05:44:29,742 - INFO - Success 7B4220D4A425489FA8948E86CAE34466
提取尺寸进度:   0%|          | 9/3744 [00:02<13:28,  4.62it/s]2026-04-07 05:44:29,751 - INFO - Success 332C16A376F84927B93DC4DCBCCB0A9F
2026-04-07 05:44:29,882 - INFO - Success CB433A75AFCD49A4983EC70E2B13C244
2026-04-07 05:44:29,884 - INFO -


✅ 所有请求完成！原始结果已存入: d:\LocalWorkSpace\20260202_museum_data_crawling\wwsdw.net\data\extracted\dimensions_20260407_054332.jsonl


In [59]:
# 读取提取的 JSONL 结果
extracted_dims = []
with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            extracted_dims.append(json.loads(line))

# 转为 DataFrame
df_dims = pd.DataFrame(extracted_dims)

# 如果你的 df_merged 原本没有 structuredDimensions 列，先初始化
if "structuredDimensions" not in df_merged.columns:
    df_merged["structuredDimensions"] = None

# 通过 id 进行更新合并
# 使用 set_index 确保对齐准确，避免产生重复行或乱序
df_merged.set_index("id", inplace=True)
df_dims.set_index("id", inplace=True)

# 将提取到的 structuredDimensions 赋值回主 df
df_merged["structuredDimensions"].update(df_dims["structuredDimensions"])

# 恢复 id 为普通列
df_merged.reset_index(inplace=True)
df_dims.reset_index(inplace=True)

print("✅ 结构化尺寸已成功合并至 df_merged 的 structuredDimensions 列！")
print("你可以运行 df_merged[['dimensionsDesc', 'structuredDimensions']].dropna(subset=['dimensionsDesc']).head() 来查看效果。")

✅ 结构化尺寸已成功合并至 df_merged 的 structuredDimensions 列！
你可以运行 df_merged[['dimensionsDesc', 'structuredDimensions']].dropna(subset=['dimensionsDesc']).head() 来查看效果。


C:\Users\NOVA\AppData\Local\Temp\ipykernel_11804\971752797.py:21: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_merged["structuredDimensions"].update(df_dims["structuredDimensions"])
C:\Users\NOVA\AppData\Local\Temp\ipykernel_11804\971752797.py:21: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[list([{'unit': 'cm', 'value': 19.2, 'deviation': None, 'range': None, 'label': '口径'}])
 list([{'unit': 'cm', 'value': 10.5, 'deviation': None, 'range'

In [60]:
df_merged.head(5)

,id,name,era,culture,time,dimensionsDesc,structuredDimensions,fullDesc,features,excavationLocation,...,yearName,material,pattern,shape,state,color,craft,raw_excavation_info,raw_size_in_desc,other_info
0,003895c08aa84c2a98d3d1696db7ffcf,新石器时代大汶口文化红陶鋬盆,新石器时代,大汶口文化,"{'year': None, 'month': None, 'day': None}",口径19.2,"[{'unit': 'cm', 'value': 19.2, 'deviation': No...",夹砂红陶，侈口，上腹斜置，折腹后收为小平底，折腹处印压出一周斜点纹，鋬手由中腹弯出，端部窄薄上翘。,"[{'label': 'material', 'value': '夹砂红陶'}, {'lab...",NaN,...,新石器时代,夹砂红陶,折腹处印压出一周斜点纹,侈口，上腹斜置，折腹后收为小平底，鋬手由中腹弯出，端部窄薄上翘,None,None,None,None,None,None
1,0043E78E1BCD4F11902DCA3C76607652,新石器时代大汶口文化褐陶鬶,新石器时代,大汶口文化,"{'year': None, 'month': None, 'day': None}",口径10.5厘米,"[{'unit': 'cm', 'value': 10.5, 'deviation': No...",None,"[{'label': 'material', 'value': '褐陶'}, {'label...",NaN,...,新石器时代,褐陶,None,鬶,None,None,None,None,None,None
2,004658C82423474EB46AA00F6BEEDAFC,新石器时代大汶口文化红陶壶,新石器时代,大汶口文化,"{'year': None, 'month': None, 'day': None}",高*口径*底径：15.3*7.5*6.2,"[{'unit': 'cm', 'value': 15.3, 'deviation': No...",None,"[{'label': 'material', 'value': '红陶'}]",NaN,...,新石器时代,红陶,None,None,None,None,None,None,None,None
3,0053DF2D67D346059E75B0729A69FE5B,新石器时代夹砂黑陶碗,新石器时代,None,"{'year': None, 'month': None, 'day': None}",口径13，底径7，高5,"[{'unit': 'cm', 'value': 13.0, 'deviation': No...",None,"[{'label': 'material', 'value': '夹砂黑陶'}, {'lab...",NaN,...,新石器时代,夹砂黑陶,None,碗,None,黑陶,None,None,None,None
4,005C79C1C40F4AB9AA9183E5B383E158,新石器时代龙山文化陶褐鬹,新石器时代,龙山文化,"{'year': None, 'month': None, 'day': None}",通高*腹围*口径*流长：39.5*37*9.1*9.9,"[{'unit': 'cm', 'value': 39.5, 'deviation': No...",None,[],NaN,...,新石器时代,None,None,None,None,None,None,None,None,None


In [61]:
import json
import asyncio
import logging
from datetime import datetime
import pandas as pd
from tqdm.asyncio import tqdm

# ==========================================
# 1. 强制空值打底（满足无信息直接全填 null 的要求）
# ==========================================
default_location = {"country": None, "province": None, "city": None, "district": None, "specificAddress": None, "coordinate": None}
default_date = {"year": None, "month": None, "day": None}

# 如果原表没有这两列，或者需要覆写，直接全量初始化为默认字典
df_merged["excavationLocation"] = [default_location.copy() for _ in range(len(df_merged))]
df_merged["excavationDate"] = [default_date.copy() for _ in range(len(df_merged))]

# ==========================================
# 2. 路径与日志配置 (沿用之前的 BASE_DIR)
# ==========================================
timestamp_exc = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_EXC_FILE = OUTPUT_DIR / f"excavation_{timestamp_exc}.jsonl"
LOG_EXC_FILE = LOG_DIR / f"api_excavation_{timestamp_exc}.log"

# 单独为出土信息开一个 FileHandler 记录日志
exc_logger = logging.getLogger("excavation")
exc_logger.setLevel(logging.INFO)
exc_handler = logging.FileHandler(LOG_EXC_FILE, encoding="utf-8")
exc_handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
exc_logger.addHandler(exc_handler)

# ==========================================
# 3. 提示词与提取逻辑
# ==========================================
SYSTEM_PROMPT_EXC = """你是一个专业的考古信息结构化提取助手。
请从用户提供的出土信息原始文本中，提取出土地点和出土日期，并严格按照 JSON 格式输出。

输出必须包含两个键："excavationLocation" 和 "excavationDate"，具体结构如下：
{
  "excavationLocation": {
    "country": "提取的国家，如无填 null",
    "province": "提取的省份，如无填 null",
    "city": "提取的城市，如无填 null",
    "district": "提取的区县，如无填 null",
    "specificAddress": "提取的具体遗址或地址，如无填 null",
    "coordinate": null
  },
  "excavationDate": {
    "year": "提取的年份(纯数字字符串或带'年'均可)，如无填 null",
    "month": "提取的月份，如无填 null",
    "day": "提取的日期，如无填 null"
  }
}
要求：
1. 只能依靠提供的文本进行提取，不知道的字段一律填 null。
2. 严格输出合法的 JSON 对象，不加任何 Markdown 标记。"""

async def extract_excavation(item_id: str, desc: str, sem: asyncio.Semaphore, file_lock: asyncio.Lock):
    async with sem:
        try:
            response = await client.chat.completions.create(
                model="deepseek-chat",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT_EXC},
                    {"role": "user", "content": f"提取以下出土信息文本:\n{desc}"}
                ],
                response_format={"type": "json_object"},
                temperature=0.1,
                timeout=45
            )
            
            extracted_json = json.loads(response.choices[0].message.content)
            
            # 增加鲁棒性，防止模型漏掉主键
            result_data = {
                "id": item_id,
                "excavationLocation": extracted_json.get("excavationLocation", default_location),
                "excavationDate": extracted_json.get("excavationDate", default_date)
            }
            
            async with file_lock:
                with open(OUTPUT_EXC_FILE, "a", encoding="utf-8") as f:
                    f.write(json.dumps(result_data, ensure_ascii=False) + "\n")
            
            exc_logger.info(f"Success {item_id}")
            
        except Exception as e:
            exc_logger.error(f"Failed {item_id}: {str(e)}")
            # 失败兜底
            async with file_lock:
                with open(OUTPUT_EXC_FILE, "a", encoding="utf-8") as f:
                    f.write(json.dumps({"id": item_id, "excavationLocation": default_location, "excavationDate": default_date}, ensure_ascii=False) + "\n")

async def run_excavation_extraction():
    # 仅筛选出 raw_excavation_info 不为空的行
    valid_mask = df_merged["raw_excavation_info"].notna() & (df_merged["raw_excavation_info"].str.strip() != "")
    tasks_data = df_merged[valid_mask][["id", "raw_excavation_info"]].to_dict("records")
    
    if not tasks_data:
        print("没有需要提取的出土信息！所有数据已默认打底为 null。")
        return
        
    print(f"准备处理 {len(tasks_data)} 条出土信息数据...")
    sem = asyncio.Semaphore(20)
    file_lock = asyncio.Lock()
    
    tasks = [extract_excavation(row["id"], row["raw_excavation_info"], sem, file_lock) for row in tasks_data]
    await tqdm.gather(*tasks, desc="出土信息提取进度")
    print(f"\n✅ 所有出土信息提取完成！原始结果已存入: {OUTPUT_EXC_FILE}")

# 执行提取
await run_excavation_extraction()

准备处理 103 条出土信息数据...


出土信息提取进度:   1%|          | 1/103 [00:02<04:02,  2.38s/it]2026-04-07 06:02:56,010 - INFO - Success CD05BD1DE800438CBE80500C3CF190B2
2026-04-07 06:02:56,021 - INFO - Success E37C9F919A2B4AE3AC9B145B29288D5D
2026-04-07 06:02:56,099 - INFO - Success 3700000201862104B100425094809093
出土信息提取进度:   4%|▍         | 4/103 [00:02<00:47,  2.07it/s]2026-04-07 06:02:56,133 - INFO - Success 3700000201862104C100420182423406
2026-04-07 06:02:56,157 - INFO - Success 3700000201862104B100424111031924
2026-04-07 06:02:56,178 - INFO - Success 3700000201862104C100425092600484
2026-04-07 06:02:56,223 - INFO - Success 06b6eb96528a4fe68887b0658eeaaf78
出土信息提取进度:   8%|▊         | 8/103 [00:02<00:19,  4.80it/s]2026-04-07 06:02:56,232 - INFO - Success 0FF15D85DEE4496C935EA878318BA2D5
2026-04-07 06:02:56,240 - INFO - Success 07b866fcbb5547bd80a464f74fa0fbc0
2026-04-07 06:02:56,255 - INFO - Success 01210ea5cbb14a6ea6de870299758873
2026-04-07 06:02:56,265 - INFO - Success 3700000201862104B100426155857937
2026-04-07 06:0


✅ 所有出土信息提取完成！原始结果已存入: d:\LocalWorkSpace\20260202_museum_data_crawling\wwsdw.net\data\extracted\excavation_20260407_060253.jsonl


In [ ]:
import os

if os.path.exists(OUTPUT_EXC_FILE):
    # 读取提取的结果
    extracted_exc = []
    with open(OUTPUT_EXC_FILE, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                extracted_exc.append(json.loads(line))
    
    if extracted_exc:
        df_exc = pd.DataFrame(extracted_exc)
        
        # 通过 id 设置索引，保证严格对齐
        df_merged.set_index("id", inplace=True)
        df_exc.set_index("id", inplace=True)
        
        # 只更新有实际提取结果的行，其余保持我们第一步设置的 null 字典
        df_merged["excavationLocation"].update(df_exc["excavationLocation"])
        df_merged["excavationDate"].update(df_exc["excavationDate"])
        
        # 恢复 id 为普通列
        df_merged.reset_index(inplace=True)
        
        print("✅ 出土信息已成功合并回 df_merged！")
else:
    print("没有生成提取文件，说明全是空值，已全部打底为 null，无需合并。")


In [64]:
default_location = {"country": None, "province": None, "city": None, "district": None, "specificAddress": None, "coordinate": None}
df_merged[df_merged["excavationLocation"]!=default_location].head(5)

,id,name,era,culture,time,dimensionsDesc,structuredDimensions,fullDesc,features,excavationLocation,...,yearName,material,pattern,shape,state,color,craft,raw_excavation_info,raw_size_in_desc,other_info
16,01210ea5cbb14a6ea6de870299758873,新石器时代龙山文化黑陶觯,新石器时代,龙山文化,"{'year': None, 'month': None, 'day': None}",高*口径*底径：14*6.6*4.1,"[{'unit': 'cm', 'value': 14.0, 'deviation': No...",新石器时代龙山文化，饮酒器。泥质黑陶，侈口圆唇，高颈，鼓腹下收，小平底。口径6.6cm、底径...,"[{'label': 'material', 'value': '泥质黑陶'}, {'lab...","{'country': '中国', 'province': '山东省', 'city': '...",...,新石器时代,泥质黑陶,None,侈口圆唇，高颈，鼓腹下收，小平底,None,黑陶,None,1979年于日照尧王城遗址发掘,口径6.6cm、底径4.1cm、高14cm，重190g,新石器时代龙山文化，饮酒器。日照市博物馆收藏。国家二级文物。
88,063f04237365413588b166fadfa28e1c,新石器时代大汶口文化刻陶文“钺”“拍”灰陶尊,新石器时代,大汶口文化,"{'year': None, 'month': None, 'day': None}",高67、口径32、壁厚2.4,"[{'unit': 'cm', 'value': 67.0, 'deviation': No...",为新石器时代大汶口文化陶尊。1960年莒县陵阳河遗址出土。通高63厘米、口径39厘米、壁厚2...,"[{'label': 'material', 'value': '夹砂黄褐陶'}, {'la...","{'country': '中国', 'province': '山东省', 'city': '...",...,新石器时代,夹砂黄褐陶,口沿下和腹下各饰两道凹弦纹，弦纹间饰有两周圆圈纹。通体饰篮纹。腹部刻有符号“钺”形，另一面刻...,折沿，深直腹，尖底,保存较好,涂有朱红,刻,1960年莒县陵阳河遗址出土,通高63厘米、口径39厘米、壁厚2.4厘米,墓主人以此来陪葬，生前可能是一位军事首领。从发掘墓葬出土的人体多为不全者来看，这意味着在50...
101,06b6eb96528a4fe68887b0658eeaaf78,新石器时代龙山文化陶鬶,新石器时代,龙山文化,"{'year': None, 'month': None, 'day': None}",通高36.8厘米，口径11.6厘米,"[{'unit': 'cm', 'value': 36.8, 'deviation': No...",侈口，口沿下饰两周不透镂孔，一道凸玄纹至流两侧并饰有乳钉纹，鸟啄形流上仰，颈呈筒形，颈腹分界...,"[{'label': 'pattern', 'value': '口沿下饰两周不透镂孔，一道凸...","{'country': None, 'province': None, 'city': '沂...",...,新石器时代,None,口沿下饰两周不透镂孔，一道凸玄纹至流两侧并饰有乳钉纹，腹上两侧饰盲鼻，并饰一周凸玄纹,侈口，鸟啄形流上仰，颈呈筒形，颈腹分界明显，乳状袋足，绳索状鋬,None,None,镂空,沂水县杨庄镇杨庄村出土,通高36.8厘米，口径11.6厘米，重1.09千克,None
102,0706381701a34e0ca5ff8bc75a9428a9,新石器时代大汶口文化彩陶漩涡纹单耳杯,新石器时代,大汶口文化,"{'year': None, 'month': None, 'day': None}",口径*底径：8.9*8,"[{'unit': 'cm', 'value': 8.9, 'deviation': Non...",该器物为泥质红陶，侈口，圆唇，高领，鋬状桥式耳，缓折腹下垂，小平底。唇内外及下腹饰黑彩一周，...,"[{'label': 'material', 'value': '泥质红陶'}, {'lab...","{'country': '中国', 'province': '山东省', 'city': '...",...,新石器时代,泥质红陶,腹部用黑彩绘成旋涡纹,侈口，圆唇，高领，鋬状桥式耳，缓折腹下垂，小平底,口沿两处略残,唇内外及下腹饰黑彩一周，腹折线以上饰红陶衣,None,广饶县傅家遗址出土，属新石器时代大汶口文化时期遗物，距今已有5000多年的历史,None,对研究鲁北地区大汶口文化提供了重要的实物资料
128,07b866fcbb5547bd80a464f74fa0fbc0,新石器时代龙山文化黑陶壶,新石器时代,龙山文化,"{'year': None, 'month': None, 'day': None}",高*口径*底径*腹径：16.6*8.4*5.5*11.9,"[{'unit': 'cm', 'value': 16.6, 'deviation': No...",新石器时代龙山文化，酒器或水器。泥质黑陶，侈口，圆唇，高领，鼓腹，腹上部饰二道凹弦纹及二个对...,"[{'label': 'material', 'value': '泥质黑陶'}, {'lab...","{'country': '中国', 'province': '山东省', 'city': '...",...,新石器时代,泥质黑陶,None,侈口，圆唇，高领，鼓腹，腹上部饰二道凹弦纹及二个对称的横耳，鼓腹下收，小平底,None,黑陶,None,采集于日照两城镇遗址,口径8.4cm、底径5.5cm、腹径11.9cm、高16.6cm，重366g,新石器时代龙山文化，酒器或水器。日照市博物馆收藏。国家二级文物。


In [65]:
default_date = {"year": None, "month": None, "day": None}
df_merged[df_merged["excavationDate"]!=default_date].head(5)

,id,name,era,culture,time,dimensionsDesc,structuredDimensions,fullDesc,features,excavationLocation,...,yearName,material,pattern,shape,state,color,craft,raw_excavation_info,raw_size_in_desc,other_info
16,01210ea5cbb14a6ea6de870299758873,新石器时代龙山文化黑陶觯,新石器时代,龙山文化,"{'year': None, 'month': None, 'day': None}",高*口径*底径：14*6.6*4.1,"[{'unit': 'cm', 'value': 14.0, 'deviation': No...",新石器时代龙山文化，饮酒器。泥质黑陶，侈口圆唇，高颈，鼓腹下收，小平底。口径6.6cm、底径...,"[{'label': 'material', 'value': '泥质黑陶'}, {'lab...","{'country': '中国', 'province': '山东省', 'city': '...",...,新石器时代,泥质黑陶,None,侈口圆唇，高颈，鼓腹下收，小平底,None,黑陶,None,1979年于日照尧王城遗址发掘,口径6.6cm、底径4.1cm、高14cm，重190g,新石器时代龙山文化，饮酒器。日照市博物馆收藏。国家二级文物。
88,063f04237365413588b166fadfa28e1c,新石器时代大汶口文化刻陶文“钺”“拍”灰陶尊,新石器时代,大汶口文化,"{'year': None, 'month': None, 'day': None}",高67、口径32、壁厚2.4,"[{'unit': 'cm', 'value': 67.0, 'deviation': No...",为新石器时代大汶口文化陶尊。1960年莒县陵阳河遗址出土。通高63厘米、口径39厘米、壁厚2...,"[{'label': 'material', 'value': '夹砂黄褐陶'}, {'la...","{'country': '中国', 'province': '山东省', 'city': '...",...,新石器时代,夹砂黄褐陶,口沿下和腹下各饰两道凹弦纹，弦纹间饰有两周圆圈纹。通体饰篮纹。腹部刻有符号“钺”形，另一面刻...,折沿，深直腹，尖底,保存较好,涂有朱红,刻,1960年莒县陵阳河遗址出土,通高63厘米、口径39厘米、壁厚2.4厘米,墓主人以此来陪葬，生前可能是一位军事首领。从发掘墓葬出土的人体多为不全者来看，这意味着在50...
150,09053083EC594D80BF074EBF18C9994B,新石器时代大汶口文化附加堆纹夹砂红陶三足鼎,None,None,"{'year': None, 'month': None, 'day': None}",None,NaN,新石器时代大汶口文化时期\r\n未定级\r\n2012年出土于界石旸里店遗址，现藏威海市文登...,"[{'label': 'material', 'value': '夹砂红褐陶'}, {'la...","{'country': None, 'province': None, 'city': '威...",...,新石器时代,夹砂红褐陶,下腹部饰1圈弦纹,尖唇，折沿，略鼓腹，裆略平，柱状足,两足略残,None,None,2012年出土于界石旸里店遗址，现藏威海市文登区博物馆。,口径10.4厘米，腹径11.6厘米，柱足长4.4厘米，通高12.2厘米。,新石器时代大汶口文化时期，未定级，明器，陶质。
203,0C1D1E0DA1424A318CA9DC8A4139FBBD,新石器时代龙山文化单耳陶杯,新石器时代,龙山文化,"{'year': None, 'month': None, 'day': None}",口径*底径*最大宽：6*4.8*13.8,"[{'unit': 'cm', 'value': 6.0, 'deviation': Non...",1989年山东邹平丁公遗址出土\r\n水器。泥质黑陶，器表经磨光，直筒形，口微侈，中部略内收...,"[{'label': 'material', 'value': '泥质黑陶'}, {'lab...","{'country': '中国', 'province': '山东', 'city': No...",...,新石器时代,泥质黑陶,None,直筒形，口微侈，中部略内收，细高体，底内凹，把手宽扁长，位于器身的底部。整体造型独特。,None,黑陶,器表经磨光,1989年山东邹平丁公遗址出土,None,水器。
266,0e17c26e26d4481c80697ebb7fff3a28,大汶口文化褐陶鬶,None,None,"{'year': None, 'month': None, 'day': None}",None,NaN,1976年9月出土于火山埠遗址\r\n 高21.6厘米,"[{'label': 'material', 'value': '褐陶'}]","{'country': None, 'province': None, 'city': No...",...,新石器时代,褐陶,None,None,None,None,None,1976年9月出土于火山埠遗址,高21.6厘米,None


In [66]:
# 为统一起见，将excavationDate为默认值的行的excavationDate字段全部置为 null
df_merged.loc[df_merged["excavationDate"] == default_date, "excavationDate"] = None

In [68]:
# 去掉以下的列："image_count" "local_image_paths" "pictureIds" "yearName" "material" "pattern" "shape" "state" "color" "craft"
df_merged.drop(columns=["image_count", "local_image_paths", "pictureIds", "yearName", "material", "pattern", "shape", "state", "color", "craft"], inplace=True)

In [69]:
final_output_path = BASE_DIR / "data" / "final_delivery.jsonl"
df_merged.to_json(final_output_path, orient="records", lines=True, force_ascii=False)